# 02 — Clean and Build the Final Dutch PEP Benchmark

This notebook combines the documented benchmark components produced or recorded in `01_data_collection_scraper.ipynb` into the final Dutch PEP benchmark snapshot.

The benchmark is constructed from:

1. the curated and manually screened benchmark v1 baseline;
2. dedicated Eerste Kamer candidate records;
3. Selenium-extracted diplomatic records;
4. small manually verified official-source additions;
5. manually reviewed CBB, CRvB, and senior-military additions;
6. manually structured and reviewed Category C political-party board records.

The notebook standardises record fields, removes leading professional and academic titles from names, applies documented cleaning adjustments, checks data quality, and saves the current reproducible benchmark snapshot.

The diplomatic source is dynamic. Accordingly, the final benchmark size is derived from the current documented collection run rather than from a historic development-run total. The generic static candidate output from notebook 01 is retained as an audit artefact but is not merged directly, because its relevant categories are already covered by the curated benchmark v1 baseline.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import re
import unicodedata

# -------------------------------------------------------------------
# Project folder setup
# -------------------------------------------------------------------

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

INPUT_DIR = PROJECT_DIR / "data" / "input"
CURATED_INTERMEDIATES_DIR = INPUT_DIR / "curated_intermediates"
OUTPUT_DIR = PROJECT_DIR / "data" / "output"
COMPONENT_DIR = OUTPUT_DIR / "collection_components"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------
# Input paths from the clean pipeline
# -------------------------------------------------------------------

CURATED_V1_PATH = (
    CURATED_INTERMEDIATES_DIR / "benchmark_v1_curated.csv"
)

SENATE_CANDIDATES_PATH = (
    COMPONENT_DIR / "senate_members_candidates.csv"
)

DIPLOMATIC_IN_SCOPE_PATH = (
    COMPONENT_DIR / "diplomatic_in_scope_candidates.csv"
)

MANUAL_SMALL_ADDITIONS_PATH = (
    INPUT_DIR / "manual_small_official_additions.csv"
)

MANUAL_ADDITIONS_PATH = (
    INPUT_DIR / "manual_main_v2_additions.xlsx"
)

CATEGORY_C_MANUAL_PATH = (
    INPUT_DIR / "category_c_party_board_manual_v3.csv"
)

KNOWN_ADJUSTMENTS_PATH = (
    OUTPUT_DIR / "known_benchmark_cleaning_adjustments.csv"
)

TAXONOMY_PATH = (
    INPUT_DIR / "taxonomy_v2_eu_aligned.xlsx"
)

FINAL_BENCHMARK_PATH = (
    OUTPUT_DIR / "main_current_benchmark_v3_extended.csv"
)

BENCHMARK_QUALITY_CHECKS_PATH = (
    OUTPUT_DIR / "benchmark_v3_quality_checks.csv"
)

BENCHMARK_CATEGORY_COUNTS_PATH = (
    OUTPUT_DIR / "benchmark_v3_category_counts.csv"
)

print("Project folder:", PROJECT_DIR)

print("\nRequired component inputs:")
print("Curated v1 baseline:", CURATED_V1_PATH.exists())
print("Senate candidates:", SENATE_CANDIDATES_PATH.exists())
print("Diplomatic in-scope candidates:", DIPLOMATIC_IN_SCOPE_PATH.exists())
print("Manual small additions:", MANUAL_SMALL_ADDITIONS_PATH.exists())
print("Manual additions workbook:", MANUAL_ADDITIONS_PATH.exists())
print("Manual Category C records:", CATEGORY_C_MANUAL_PATH.exists())
print("Known cleaning adjustments:", KNOWN_ADJUSTMENTS_PATH.exists())
print("Taxonomy workbook:", TAXONOMY_PATH.exists())

print("\nFinal benchmark output:")
print(FINAL_BENCHMARK_PATH)

Project folder: c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean

Required component inputs:
Curated v1 baseline: True
Senate candidates: True
Diplomatic in-scope candidates: True
Manual small additions: True
Manual additions workbook: True
Manual Category C records: True
Known cleaning adjustments: True
Taxonomy workbook: True

Final benchmark output:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\main_current_benchmark_v3_extended.csv


## 1. Load all benchmark components

The component files are loaded without modifying the original source files.

At this stage, the files may use different column structures because they originate from different collection routes. The next sections will standardise them into one common benchmark schema.

In [2]:
# -------------------------------------------------------------------
# Helper function for loading CSV files with common encodings
# -------------------------------------------------------------------

def read_csv_with_fallback_encodings(file_path):
    """
    Load a CSV file using common encodings.

    Some CSV files saved through Excel use cp1252 rather than UTF-8.
    """
    possible_encodings = [
        "utf-8",
        "utf-8-sig",
        "cp1252",
        "latin1"
    ]

    last_error = None

    for encoding in possible_encodings:
        try:
            dataframe = pd.read_csv(
                file_path,
                encoding=encoding
            )

            print(f"Loaded {file_path.name} using encoding: {encoding}")

            return dataframe, encoding

        except UnicodeDecodeError as error:
            last_error = error

    raise ValueError(
        f"Could not load {file_path.name}. "
        f"Last encoding error: {last_error}"
    )


# -------------------------------------------------------------------
# Load CSV component files
# -------------------------------------------------------------------

curated_v1_df, curated_v1_encoding = (
    read_csv_with_fallback_encodings(CURATED_V1_PATH)
)

senate_df, senate_encoding = (
    read_csv_with_fallback_encodings(SENATE_CANDIDATES_PATH)
)

diplomatic_df, diplomatic_encoding = (
    read_csv_with_fallback_encodings(DIPLOMATIC_IN_SCOPE_PATH)
)

manual_small_df, manual_small_encoding = (
    read_csv_with_fallback_encodings(MANUAL_SMALL_ADDITIONS_PATH)
)

category_c_df, category_c_encoding = (
    read_csv_with_fallback_encodings(CATEGORY_C_MANUAL_PATH)
)

known_adjustments_df, known_adjustments_encoding = (
    read_csv_with_fallback_encodings(KNOWN_ADJUSTMENTS_PATH)
)

# -------------------------------------------------------------------
# Load manual additions workbook sheets
# -------------------------------------------------------------------

manual_additions_excel = pd.ExcelFile(
    MANUAL_ADDITIONS_PATH
)

manual_additions_by_sheet = {
    sheet_name: pd.read_excel(
        MANUAL_ADDITIONS_PATH,
        sheet_name=sheet_name
    )
    for sheet_name in manual_additions_excel.sheet_names
}

print("\nComponent row counts:")
print("Curated v1 baseline:", len(curated_v1_df))
print("Senate candidates:", len(senate_df))
print("Diplomatic in-scope candidates:", len(diplomatic_df))
print("Manual small additions:", len(manual_small_df))
print("Manual Category C records:", len(category_c_df))

print("\nManual workbook sheets and row counts:")
for sheet_name, sheet_df in manual_additions_by_sheet.items():
    print(f"{sheet_name}: {len(sheet_df)} rows")

print("\nKnown cleaning adjustments:")
display(known_adjustments_df)

Loaded benchmark_v1_curated.csv using encoding: utf-8
Loaded senate_members_candidates.csv using encoding: utf-8
Loaded diplomatic_in_scope_candidates.csv using encoding: utf-8
Loaded manual_small_official_additions.csv using encoding: utf-8
Loaded category_c_party_board_manual_v3.csv using encoding: utf-8
Loaded known_benchmark_cleaning_adjustments.csv using encoding: utf-8

Component row counts:
Curated v1 baseline: 286
Senate candidates: 75
Diplomatic in-scope candidates: 89
Manual small additions: 9
Manual Category C records: 79

Manual workbook sheets and row counts:
cbb: 46 rows
crvb: 88 rows
senior_military: 5 rows

Known cleaning adjustments:


,adjustment_type,person_name,taxonomy_category,reason,records_removed
0,exclude_out_of_scope_role,drs. E. Storm,high_judiciary_crvb,Listed as a non-judicial CRvB board member. Th...,1
1,deduplicate_person_category,R.W.L. Koopmans,high_judiciary_crvb,The same person appears twice in the CRvB sour...,1


In [3]:
# -------------------------------------------------------------------
# Inspect component schemas before standardisation
# -------------------------------------------------------------------

component_schemas = {
    "curated_v1_baseline": curated_v1_df,
    "senate_candidates": senate_df,
    "diplomatic_in_scope": diplomatic_df,
    "manual_small_additions": manual_small_df,
    "category_c_manual": category_c_df
}

for component_name, component_df in component_schemas.items():
    print(f"\n{component_name}")
    print("-" * len(component_name))
    print(component_df.columns.tolist())

print("\nManual workbook schemas:")
for sheet_name, sheet_df in manual_additions_by_sheet.items():
    print(f"\nmanual_additions/{sheet_name}")
    print("-" * (len(sheet_name) + 17))
    print(sheet_df.columns.tolist())


curated_v1_baseline
-------------------
['person_name', 'normalised_name', 'role_title', 'role_nl', 'role_eng', 'taxonomy_category', 'official_function', 'institutional_level', 'institution', 'source_url', 'page_title', 'extractor_used', 'access_timestamp_utc', 'validation_status', 'included_in_main_benchmark', 'operationalisation_status', 'source_row', 'category', 'cleaning_notes']

senate_candidates
-----------------
['category', 'source_row', 'role_nl', 'role_eng', 'person_name', 'role_title', 'institution', 'source_url', 'page_title', 'extractor_used', 'access_timestamp_utc', 'senate_listing_name', 'official_profile_name', 'official_profile_url', 'profile_fetch_status', 'profile_name_extraction_status', 'evidence_text']

diplomatic_in_scope
-------------------
['page_number', 'category', 'source_row', 'role_nl', 'role_eng', 'person_name', 'role_title', 'taxonomy_category', 'include_in_main_benchmark', 'institution', 'source_url', 'page_title', 'extractor_used', 'access_timestamp_u

In [4]:
# -------------------------------------------------------------------
# Inspect the manual judiciary-addition workbook schemas in full
# -------------------------------------------------------------------
# The CBB and CRvB sheets may contain more fields than the military
# sheet. We inspect their exact columns before standardising them into
# the final common benchmark structure.

for sheet_name in ["cbb", "crvb", "senior_military"]:
    sheet_df = manual_additions_by_sheet[sheet_name]

    print(f"\n{'=' * 70}")
    print(f"Sheet: {sheet_name}")
    print(f"{'=' * 70}")

    print("\nColumns:")
    for column_name in sheet_df.columns:
        print(f"- {column_name}")

    print("\nFirst two rows:")
    display(sheet_df.head(2))


Sheet: cbb

Columns:
- person_name
- role_title
- role_nl
- role_eng
- taxonomy_category
- official_function
- institutional_level
- institution
- source_url
- page_title
- extractor_used
- validation_status
- included_in_main_benchmark
- operationalisation_status
- category
- cleaning_notes

First two rows:


,person_name,role_title,role_nl,role_eng,taxonomy_category,official_function,institutional_level,institution,source_url,page_title,extractor_used,validation_status,included_in_main_benchmark,operationalisation_status,category,cleaning_notes
0,mr. J.L.W. Aerts,Rechter,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mr. R.C. Stam,Rechter,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Sheet: crvb

Columns:
- person_name
- role_title
- role_nl
- role_eng
- taxonomy_category
- official_function
- institutional_level
- institution
- source_url
- page_title
- extractor_used
- validation_status
- included_in_main_benchmark
- operationalisation_status
- category
- cleaning_notes

First two rows:


,person_name,role_title,role_nl,role_eng,taxonomy_category,official_function,institutional_level,institution,source_url,page_title,extractor_used,validation_status,included_in_main_benchmark,operationalisation_status,category,cleaning_notes
0,mr. drs. J.C. Boeree,President,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mr. R.W.L. Koopmans,Interim rechterlijk bestuurslid,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Sheet: senior_military

Columns:
- person_name
- role_title
- source_url

First two rows:


,person_name,role_title,source_url
0,Onno Eichelsheim,Commandant der Strijdkrachten,https://www.defensie.nl/organisatie/bestuursst...
1,Harold Liebregs,Commandant Zeestrijdkrachten,https://www.defensie.nl/organisatie/marine/cszk


## 2. Standardise all components into one benchmark schema

The benchmark components originate from different collection routes and therefore contain different columns.

This section converts each component into a common benchmark structure. Missing shared metadata for CBB, CRvB, senior military, Senate, diplomatic, and Category C records is filled from the documented Dutch taxonomy and from the known collection method.

The original component files are not modified. A combined pre-cleaning dataset is saved as an audit artefact before the final cleaning adjustments are applied.

In [5]:
# -------------------------------------------------------------------
# Helper function for saving CSV files safely
# -------------------------------------------------------------------
# If an earlier output file is open in Excel, the function writes a
# timestamped fallback file instead of stopping the notebook.

def write_csv_safely(dataframe, output_path):
    """
    Save a CSV file with UTF-8 encoding.

    When the intended output file is locked, save a timestamped
    alternative file instead.
    """
    try:
        dataframe.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        return output_path

    except PermissionError:
        timestamp = datetime.now(
            timezone.utc
        ).strftime("%Y%m%d_%H%M%S")

        fallback_path = output_path.with_name(
            f"{output_path.stem}_{timestamp}{output_path.suffix}"
        )

        print(
            "Warning: the output file appears to be open. "
            f"Writing to {fallback_path.name} instead."
        )

        dataframe.to_csv(
            fallback_path,
            index=False,
            encoding="utf-8-sig"
        )

        return fallback_path

In [6]:
# -------------------------------------------------------------------
# Standardise all benchmark components into one common schema
# -------------------------------------------------------------------
# The different collection routes use different column structures.
# This section creates one consistent benchmark table while preserving
# source-component provenance for every record.

RAW_COMPONENT_BENCHMARK_PATH = (
    OUTPUT_DIR / "benchmark_v3_components_before_cleaning.csv"
)

# -------------------------------------------------------------------
# Load Dutch taxonomy metadata for category-level enrichment
# -------------------------------------------------------------------

nl_taxonomy_df = pd.read_excel(
    TAXONOMY_PATH,
    sheet_name="nl_source_mapping_detail"
)

nl_taxonomy_df.columns = (
    nl_taxonomy_df.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

def get_taxonomy_value(taxonomy_category, column_name):
    """
    Return a value from the Dutch taxonomy mapping for one local category.

    Returns pd.NA when no value is available.
    """
    matching_rows = nl_taxonomy_df[
        nl_taxonomy_df["local_taxonomy_category"]
        .astype("string")
        .eq(taxonomy_category)
    ]

    if matching_rows.empty:
        return pd.NA

    value = matching_rows.iloc[0].get(column_name, pd.NA)

    if pd.isna(value):
        return pd.NA

    return value


# -------------------------------------------------------------------
# Define the final common benchmark columns
# -------------------------------------------------------------------

FINAL_BENCHMARK_COLUMNS = [
    "person_name",
    "normalised_name",
    "role_title",
    "role_nl",
    "role_eng",
    "taxonomy_category",
    "canonical_category",
    "official_function",
    "institutional_level",
    "institution",
    "party_name",
    "source_url",
    "page_title",
    "extractor_used",
    "access_timestamp_utc",
    "validation_status",
    "included_in_main_benchmark",
    "operationalisation_status",
    "source_row",
    "category",
    "source_component",
    "evidence_text",
    "review_notes",
    "cleaning_notes"
]


def blank_or_missing_mask(series):
    """
    Return True for missing or empty text values.
    """
    return (
        series.isna()
        | series.astype("string").str.strip().eq("")
    ).fillna(True).astype(bool)


def ensure_common_columns(dataframe):
    """
    Add any missing final benchmark columns as empty values.
    """
    dataframe = dataframe.copy()

    for column_name in FINAL_BENCHMARK_COLUMNS:
        if column_name not in dataframe.columns:
            dataframe[column_name] = pd.NA

    return dataframe


def fill_missing_value(dataframe, column_name, value):
    """
    Fill a column only where it is empty.

    Existing component-specific values are never overwritten.
    """
    if pd.isna(value):
        return dataframe

    missing_mask = blank_or_missing_mask(
        dataframe[column_name]
    )

    dataframe.loc[missing_mask, column_name] = value

    return dataframe


# -------------------------------------------------------------------
# Category-level defaults from the Dutch taxonomy and methodology
# -------------------------------------------------------------------

CATEGORY_CODE_MAP = {
    "head_of_state": "A",
    "national_executive": "A",
    "senate_members": "B",
    "house_members": "B",
    "registered_party_board": "C",
    "high_judiciary_rvs": "D",
    "high_judiciary_hr": "D",
    "high_judiciary_cbb": "D",
    "high_judiciary_crvb": "D",
    "court_of_audit": "E",
    "central_bank_board": "E",
    "ambassadors": "F",
    "charges_d_affaires": "F",
    "senior_military": "F"
}

CATEGORY_FUNCTION_DEFAULTS = {
    "senate_members": {
        "official_function": "Member of the Senate",
        "institutional_level": "National legislature",
        "institution": "Eerste Kamer der Staten-Generaal",
        "validation_status": "confirmed",
        "included_in_main_benchmark": "Yes",
        "operationalisation_status": "Direct"
    },
    "ambassadors": {
        "official_function": "Ambassador",
        "institutional_level": "National diplomatic service",
        "institution": "Ministerie van Buitenlandse Zaken",
        "validation_status": "confirmed",
        "included_in_main_benchmark": "Yes",
        "operationalisation_status": "Direct"
    },
    "charges_d_affaires": {
        "official_function": "Chargé d'affaires",
        "institutional_level": "National diplomatic service",
        "institution": "Ministerie van Buitenlandse Zaken",
        "validation_status": "confirmed",
        "included_in_main_benchmark": "Yes",
        "operationalisation_status": "Direct"
    },
    "high_judiciary_cbb": {
        "official_function": (
            "Member with judicial duties of the College of Appeal "
            "for Business"
        ),
        "institutional_level": "National high judiciary",
        "institution": (
            "College van Beroep voor het bedrijfsleven"
        ),
        "validation_status": "confirmed",
        "included_in_main_benchmark": "Yes",
        "operationalisation_status": "Direct"
    },
    "high_judiciary_crvb": {
        "official_function": (
            "Member with judicial duties of the Central Appeals Tribunal"
        ),
        "institutional_level": "National high judiciary",
        "institution": "Centrale Raad van Beroep",
        "validation_status": "confirmed",
        "included_in_main_benchmark": "Yes",
        "operationalisation_status": "Direct"
    },
    "senior_military": {
        "official_function": "Senior military commander",
        "institutional_level": "National military command",
        "institution": "Ministerie van Defensie",
        "validation_status": "confirmed",
        "included_in_main_benchmark": "Yes",
        "operationalisation_status": "Direct"
    },
    "registered_party_board": {
        "official_function": (
            "Member of the governing body of a political party"
        ),
        "institutional_level": "National political party",
        "validation_status": "confirmed",
        "included_in_main_benchmark": "Yes",
        "operationalisation_status": "Extended"
    }
}


def apply_taxonomy_defaults(dataframe):
    """
    Fill missing category-level metadata using taxonomy values and
    documented benchmark-method defaults.
    """
    dataframe = dataframe.copy()

    for taxonomy_category in (
        dataframe["taxonomy_category"]
        .dropna()
        .astype(str)
        .unique()
    ):
        category_mask = (
            dataframe["taxonomy_category"]
            .astype("string")
            .eq(taxonomy_category)
            .fillna(False)
        )

        # Fill taxonomy-linked values where available.
        taxonomy_fields = [
            "role_nl",
            "role_eng",
            "source_url",
            "canonical_category"
        ]

        for field_name in taxonomy_fields:
            taxonomy_value = get_taxonomy_value(
                taxonomy_category,
                field_name
            )

            if field_name == "canonical_category":
                output_column = "canonical_category"
            else:
                output_column = field_name

            missing_mask = (
                category_mask
                & blank_or_missing_mask(
                    dataframe[output_column]
                )
            )

            if not pd.isna(taxonomy_value):
                dataframe.loc[
                    missing_mask,
                    output_column
                ] = taxonomy_value

        # Fill the category-code letter.
        category_code = CATEGORY_CODE_MAP.get(
            taxonomy_category,
            pd.NA
        )

        if not pd.isna(category_code):
            missing_mask = (
                category_mask
                & blank_or_missing_mask(
                    dataframe["category"]
                )
            )

            dataframe.loc[
                missing_mask,
                "category"
            ] = category_code

        # Fill documented category-level defaults.
        category_defaults = CATEGORY_FUNCTION_DEFAULTS.get(
            taxonomy_category,
            {}
        )

        for column_name, default_value in category_defaults.items():
            missing_mask = (
                category_mask
                & blank_or_missing_mask(
                    dataframe[column_name]
                )
            )

            dataframe.loc[
                missing_mask,
                column_name
            ] = default_value

    return dataframe


def prepare_component(dataframe, component_name):
    """
    Add a component label and guarantee the final shared schema.
    """
    dataframe = dataframe.copy()

    dataframe["source_component"] = component_name

    dataframe = ensure_common_columns(dataframe)

    return dataframe[FINAL_BENCHMARK_COLUMNS].copy()


# -------------------------------------------------------------------
# Prepare individual components
# -------------------------------------------------------------------

# 1. Frozen reviewed baseline benchmark.
curated_v1_component_df = prepare_component(
    curated_v1_df,
    "curated_v1_baseline"
)

# 2. Dedicated official Eerste Kamer extraction.
senate_component_df = senate_df.copy()
senate_component_df["taxonomy_category"] = "senate_members"
senate_component_df["included_in_main_benchmark"] = "Yes"
senate_component_df["validation_status"] = "confirmed"
senate_component_df["operationalisation_status"] = "Direct"
senate_component_df["cleaning_notes"] = (
    "Source-specific extraction from the official Eerste Kamer list "
    "page, enriched with profile-derived full names where available. "
    "The original list-page name is retained in evidence_text."
)
senate_component_df = prepare_component(
    senate_component_df,
    "senate_source_specific_extraction"
)

# 3. Selenium-extracted ambassadors and chargés d'affaires.
diplomatic_component_df = diplomatic_df.copy()
diplomatic_component_df = diplomatic_component_df.rename(
    columns={
        "include_in_main_benchmark": "included_in_main_benchmark"
    }
)
diplomatic_component_df["validation_status"] = "confirmed"
diplomatic_component_df["operationalisation_status"] = "Direct"
diplomatic_component_df["cleaning_notes"] = (
    "Selenium-rendered extraction from the official Rijksoverheid "
    "diplomatic source."
)
diplomatic_component_df = prepare_component(
    diplomatic_component_df,
    "diplomatic_selenium_extraction"
)

# 4. Small manually verified official-source additions.
manual_small_component_df = prepare_component(
    manual_small_df,
    "manual_small_official_additions"
)

# 5. Manually reviewed CBB and CRvB additions.
cbb_component_df = manual_additions_by_sheet["cbb"].copy()
cbb_component_df["taxonomy_category"] = "high_judiciary_cbb"
cbb_component_df["extractor_used"] = (
    "manual_official_source_addition"
)
cbb_component_df["cleaning_notes"] = (
    "Reviewed official-source addition from the CBB manual additions sheet."
)
cbb_component_df = prepare_component(
    cbb_component_df,
    "manual_cbb_additions"
)

crvb_component_df = manual_additions_by_sheet["crvb"].copy()
crvb_component_df["taxonomy_category"] = "high_judiciary_crvb"
crvb_component_df["extractor_used"] = (
    "manual_official_source_addition"
)
crvb_component_df["cleaning_notes"] = (
    "Reviewed official-source addition from the CRvB manual additions sheet."
)
crvb_component_df = prepare_component(
    crvb_component_df,
    "manual_crvb_additions"
)

# 6. Manually reviewed senior-military additions.
military_component_df = manual_additions_by_sheet[
    "senior_military"
].copy()
military_component_df["taxonomy_category"] = "senior_military"
military_component_df["extractor_used"] = (
    "manual_official_source_addition"
)
military_component_df["cleaning_notes"] = (
    "Reviewed official-source addition from the senior-military sheet."
)
military_component_df = prepare_component(
    military_component_df,
    "manual_senior_military_additions"
)

# 7. Manually coded national party-board records.
category_c_component_df = category_c_df.copy()
category_c_component_df["institution"] = (
    category_c_component_df["party_name"]
)
category_c_component_df["extractor_used"] = (
    category_c_component_df["extraction_method"]
)
category_c_component_df["validation_status"] = "confirmed"
category_c_component_df["included_in_main_benchmark"] = "Yes"
category_c_component_df["operationalisation_status"] = "Extended"
category_c_component_df["cleaning_notes"] = (
    "Manual structured coding and review from an official national "
    "party-board source."
)
category_c_component_df["review_notes"] = (
    category_c_component_df["review_notes"]
)
category_c_component_df = prepare_component(
    category_c_component_df,
    "manual_category_c_party_board_coding"
)

# -------------------------------------------------------------------
# Combine components and enrich missing metadata
# -------------------------------------------------------------------

raw_benchmark_components_df = pd.concat(
    [
        curated_v1_component_df,
        senate_component_df,
        diplomatic_component_df,
        manual_small_component_df,
        cbb_component_df,
        crvb_component_df,
        military_component_df,
        category_c_component_df
    ],
    ignore_index=True
)

raw_benchmark_components_df = apply_taxonomy_defaults(
    raw_benchmark_components_df
)

# Preserve category-C institution names from the manual party file.
category_c_mask = (
    raw_benchmark_components_df["taxonomy_category"]
    .astype("string")
    .eq("registered_party_board")
    .fillna(False)
)

raw_benchmark_components_df.loc[
    category_c_mask
    & blank_or_missing_mask(
        raw_benchmark_components_df["institution"]
    ),
    "institution"
] = raw_benchmark_components_df.loc[
    category_c_mask
    & blank_or_missing_mask(
        raw_benchmark_components_df["institution"]
    ),
    "party_name"
]

# Save the raw combined dataset before title cleaning and documented
# exclusions/deduplication are applied in the next section.
raw_component_output_path = write_csv_safely(
    raw_benchmark_components_df,
    RAW_COMPONENT_BENCHMARK_PATH
)

print("Saved raw combined component dataset:")
print(raw_component_output_path)

print("\nRaw component records:", len(raw_benchmark_components_df))

print("\nRecords by source component:")
display(
    raw_benchmark_components_df[
        "source_component"
    ]
    .value_counts()
    .rename_axis("source_component")
    .reset_index(name="record_count")
)

print("\nRecords by taxonomy category:")
display(
    raw_benchmark_components_df[
        "taxonomy_category"
    ]
    .value_counts(dropna=False)
    .rename_axis("taxonomy_category")
    .reset_index(name="record_count")
)

missing_category_rows = raw_benchmark_components_df[
    blank_or_missing_mask(
        raw_benchmark_components_df["taxonomy_category"]
    )
].copy()

print("\nRows without a taxonomy category:", len(missing_category_rows))

if len(missing_category_rows) > 0:
    display(
        missing_category_rows[
            [
                "person_name",
                "role_title",
                "source_component",
                "source_url"
            ]
        ]
    )

Saved raw combined component dataset:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\benchmark_v3_components_before_cleaning.csv

Raw component records: 677

Records by source component:


,source_component,record_count
0,curated_v1_baseline,286
1,diplomatic_selenium_extraction,89
2,manual_crvb_additions,88
3,manual_category_c_party_board_coding,79
4,senate_source_specific_extraction,75
5,manual_cbb_additions,46
6,manual_small_official_additions,9
7,manual_senior_military_additions,5



Records by taxonomy category:


,taxonomy_category,record_count
0,house_members,150
1,high_judiciary_crvb,88
2,ambassadors,82
3,registered_party_board,79
4,senate_members,75
5,high_judiciary_rvs,74
6,high_judiciary_cbb,46
7,high_judiciary_hr,34
8,national_executive,28
9,charges_d_affaires,7



Rows without a taxonomy category: 0


## 3. Clean names and apply documented benchmark adjustments

This section standardises names before the final benchmark is created.

Leading academic, legal, and military titles are removed from `person_name`, while the original source name is retained for auditability. A normalised name is then generated for matching.

The section also applies the two documented benchmark adjustments:

1. exclusion of the non-judicial CRvB board member;
2. removal of duplicate person-category records, including the duplicated CRvB record for R.W.L. Koopmans.

In [7]:
# -------------------------------------------------------------------
# Clean person names and apply documented benchmark adjustments
# -------------------------------------------------------------------

TITLE_CLEANING_AUDIT_PATH = (
    OUTPUT_DIR / "benchmark_v3_title_cleaning_audit.csv"
)

CLEANING_ADJUSTMENT_AUDIT_PATH = (
    OUTPUT_DIR / "benchmark_v3_cleaning_adjustment_audit.csv"
)

# -------------------------------------------------------------------
# Name-cleaning helper functions
# -------------------------------------------------------------------

# -------------------------------------------------------------------
# Safer leading-title pattern
# -------------------------------------------------------------------
# A title must be followed by either a period or whitespace.
# This prevents accidental changes such as:
# Ingrid Michon-Derkzen -> rid Michon-Derkzen

leading_title_pattern = re.compile(
    r"""
    ^\s*
    (?:
        (?:em|prof|mr|mrs|drs|dr|ir|ing|mw|mevr|dhr)\.\s*
        |
        (?:em|prof|mr|mrs|drs|dr|ir|ing|mw|mevr|dhr)\s+
        |
        professor\s+
        |
        mevrouw\s+
        |
        meneer\s+
        |
        bgen(?:\s*\([^)]+\))?\s+
    )+
    """,
    flags=re.IGNORECASE | re.VERBOSE
)

# -------------------------------------------------------------------
# Safe trailing-degree pattern
# -------------------------------------------------------------------
# A degree may only be removed when it is separated from the person's
# name by whitespace or a comma. This prevents surnames ending in
# "ma" from being changed, for example:
#
# Sjoerd Sjoerdsma -> Sjoerd Sjoerdsma
# Femke Wiersma -> Femke Wiersma
# R. van Aelst-den Uijl MA -> R. van Aelst-den Uijl

trailing_degree_pattern = re.compile(
    r"""
    (?:
        (?:\s*,\s*|\s+)
        (?:
            LL\.?\s?M\.? |
            MBA |
            M\.?Sc\.? |
            MSc |
            MDR |
            MMC |
            MA |
            BA |
            B\.?Sc\.?
        )
    )+
    \s*$
    """,
    flags=re.IGNORECASE | re.VERBOSE
)


def clean_person_name_titles(name):
    """
    Remove leading titles and trailing degree suffixes from a name.

    Examples:
    - prof. dr. E.B. van Apeldoorn -> E.B. van Apeldoorn
    - dr. mr. J. Bakker-Klein -> J. Bakker-Klein
    - Bgen (b.d.) drs. A.J.A. Beukering -> A.J.A. Beukering
    - M. Karaaslan MMC -> M. Karaaslan
    """
    if pd.isna(name):
        return pd.NA

    cleaned_name = str(name).replace("\xa0", " ").strip()

    previous_name = None

    while previous_name != cleaned_name:
        previous_name = cleaned_name

        cleaned_name = leading_title_pattern.sub(
            "",
            cleaned_name
        ).strip()

    cleaned_name = trailing_degree_pattern.sub(
        "",
        cleaned_name
    ).strip()

    cleaned_name = re.sub(
        r"\s+",
        " ",
        cleaned_name
    ).strip()

    return cleaned_name


def normalise_for_matching(name):
    """
    Create a comparison-friendly name field.

    The original clean name remains unchanged. This normalised version
    removes accents, punctuation, and repeated whitespace.
    """
    if pd.isna(name):
        return ""

    name = str(name).strip().lower()

    name = unicodedata.normalize("NFKD", name)

    name = "".join(
        character
        for character in name
        if not unicodedata.combining(character)
    )

    name = re.sub(
        r"[^a-z0-9\s]",
        " ",
        name
    )

    name = re.sub(
        r"\s+",
        " ",
        name
    ).strip()

    return name


# -------------------------------------------------------------------
# Preserve source names and apply title/degree cleaning
# -------------------------------------------------------------------

benchmark_v3_working_df = raw_benchmark_components_df.copy()

benchmark_v3_working_df["person_name_original"] = (
    benchmark_v3_working_df["person_name"]
)

benchmark_v3_working_df["person_name_before_title_cleaning"] = (
    benchmark_v3_working_df["person_name"]
)

benchmark_v3_working_df["person_name"] = (
    benchmark_v3_working_df["person_name"]
    .apply(clean_person_name_titles)
)

benchmark_v3_working_df["normalised_name"] = (
    benchmark_v3_working_df["person_name"]
    .apply(normalise_for_matching)
)

# Create a title-cleaning audit before removing any rows.
title_cleaning_audit_df = benchmark_v3_working_df[
    benchmark_v3_working_df[
        "person_name_before_title_cleaning"
    ].astype("string")
    != benchmark_v3_working_df[
        "person_name"
    ].astype("string")
].copy()

title_cleaning_audit_output_path = write_csv_safely(
    title_cleaning_audit_df,
    TITLE_CLEANING_AUDIT_PATH
)

print("Rows changed by title/degree cleaning:", len(title_cleaning_audit_df))


# -------------------------------------------------------------------
# Apply documented exclusion: non-judicial CRvB board member
# -------------------------------------------------------------------
# The source spelling may include different titles or initials.
# Therefore, the exclusion is linked to the manual CRvB component,
# the CRvB taxonomy category, and the surname Storm.

storm_exclusion_mask = (
    benchmark_v3_working_df["taxonomy_category"]
    .astype("string")
    .eq("high_judiciary_crvb")
    .fillna(False)
    &
    benchmark_v3_working_df["source_component"]
    .astype("string")
    .eq("manual_crvb_additions")
    .fillna(False)
    &
    benchmark_v3_working_df["person_name_original"]
    .astype("string")
    .str.contains(r"\bStorm\b", case=False, na=False, regex=True)
)

storm_rows_found = int(storm_exclusion_mask.sum())

if storm_rows_found != 1:
    raise ValueError(
        "Expected exactly one non-judicial CRvB Storm record for exclusion, "
        f"but found {storm_rows_found}. Review the diagnostic output."
    )
storm_excluded_df = benchmark_v3_working_df[
    storm_exclusion_mask
].copy()

benchmark_v3_working_df = benchmark_v3_working_df[
    ~storm_exclusion_mask
].copy()

# -------------------------------------------------------------------
# Apply person-category deduplication
# -------------------------------------------------------------------
# One person may have several role titles, but the benchmark uses one
# record per person per taxonomy category.

duplicate_person_category_mask = (
    benchmark_v3_working_df
    .duplicated(
        subset=[
            "normalised_name",
            "taxonomy_category"
        ],
        keep="first"
    )
)

duplicate_person_category_df = benchmark_v3_working_df[
    duplicate_person_category_mask
].copy()

benchmark_v3_working_df = benchmark_v3_working_df[
    ~duplicate_person_category_mask
].copy()

# -------------------------------------------------------------------
# Create an audit table for all removed records
# -------------------------------------------------------------------

storm_audit_df = storm_excluded_df.copy()

storm_audit_df["adjustment_type"] = (
    "exclude_out_of_scope_role"
)

storm_audit_df["adjustment_reason"] = (
    "Non-judicial CRvB board member excluded because the benchmark "
    "includes only members with judicial duties."
)

duplicate_audit_df = duplicate_person_category_df.copy()

duplicate_audit_df["adjustment_type"] = (
    "deduplicate_person_category"
)

duplicate_audit_df["adjustment_reason"] = (
    "Duplicate person-category record removed. The final benchmark "
    "retains one record per person and taxonomy category."
)

cleaning_adjustment_audit_df = pd.concat(
    [
        storm_audit_df,
        duplicate_audit_df
    ],
    ignore_index=True,
    sort=False
)

cleaning_adjustment_audit_output_path = write_csv_safely(
    cleaning_adjustment_audit_df,
    CLEANING_ADJUSTMENT_AUDIT_PATH
)

# -------------------------------------------------------------------
# Assign final benchmark identifiers
# -------------------------------------------------------------------

benchmark_v3_working_df = (
    benchmark_v3_working_df
    .sort_values(
        [
            "taxonomy_category",
            "normalised_name",
            "institution"
        ],
        na_position="last"
    )
    .reset_index(drop=True)
)

benchmark_v3_working_df.insert(
    0,
    "benchmark_record_id",
    range(1, len(benchmark_v3_working_df) + 1)
)

benchmark_v3_working_df["benchmark_version"] = "v3_extended"

print("\nCleaning summary:")
print("Raw component records:", len(raw_benchmark_components_df))
print("Excluded non-judicial CRvB rows:", len(storm_excluded_df))
print(
    "Duplicate person-category rows removed:",
    len(duplicate_person_category_df)
)
print("Final benchmark records:", len(benchmark_v3_working_df))

print("\nSaved title-cleaning audit:")
print(title_cleaning_audit_output_path)

print("\nSaved cleaning-adjustment audit:")
print(cleaning_adjustment_audit_output_path)

print("\nRemoved-record audit:")
display(
    cleaning_adjustment_audit_df[
        [
            "person_name_original",
            "person_name",
            "taxonomy_category",
            "role_title",
            "source_component",
            "adjustment_type",
            "adjustment_reason"
        ]
    ]
)


# -------------------------------------------------------------------
# Validate that only the documented adjustments changed row counts.
# -------------------------------------------------------------------

expected_final_snapshot_record_count = (
    len(raw_benchmark_components_df)
    - len(cleaning_adjustment_audit_df)
)

if len(benchmark_v3_working_df) != expected_final_snapshot_record_count:
    raise ValueError(
        "The cleaned benchmark size does not equal the raw component "
        "size minus the documented cleaning adjustments. "
        f"Expected {expected_final_snapshot_record_count}, but found "
        f"{len(benchmark_v3_working_df)}."
    )

benchmark_v3_working_df["benchmark_version"] = (
    "v3_live_source_snapshot"
)

print(
    "\nValidation passed: the cleaned benchmark contains "
    f"{len(benchmark_v3_working_df)} records in the current "
    "live-source snapshot."
)


Rows changed by title/degree cleaning: 210

Cleaning summary:
Raw component records: 677
Excluded non-judicial CRvB rows: 1
Duplicate person-category rows removed: 1
Final benchmark records: 675

Saved title-cleaning audit:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\benchmark_v3_title_cleaning_audit.csv

Saved cleaning-adjustment audit:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\benchmark_v3_cleaning_adjustment_audit.csv

Removed-record audit:


,person_name_original,person_name,taxonomy_category,role_title,source_component,adjustment_type,adjustment_reason
0,drs. E. Storm,E. Storm,high_judiciary_crvb,Niet-rechterlijk bestuurslid,manual_crvb_additions,exclude_out_of_scope_role,Non-judicial CRvB board member excluded becaus...
1,mr. R.W.L. Koopmans,R.W.L. Koopmans,high_judiciary_crvb,Raadsheer-plaatsvervanger,manual_crvb_additions,deduplicate_person_category,Duplicate person-category record removed. The ...



Validation passed: the cleaned benchmark contains 675 records in the current live-source snapshot.


### Required run order

Run this notebook from a **fresh kernel** and execute all cells from top to bottom.

The Volt metadata correction and the live-snapshot validation must run before the final quality checks and export. The notebook has been ordered accordingly.


### Complete missing Volt party metadata

Four reviewed Category C records originate from the official Volt Nederland board page but have an empty party-name field in the manual input.

The party name and institution are completed from the official source domain. This correction affects metadata only; it does not add, remove, or change any people or roles.

In [8]:
# -------------------------------------------------------------------
# Fill missing party and institution metadata for Volt records
# -------------------------------------------------------------------
# These four records use the official Volt Nederland board page as
# their source, but the party_name field was empty in the manual input.

volt_missing_metadata_mask = (
    benchmark_v3_working_df["taxonomy_category"]
    .astype("string")
    .eq("registered_party_board")
    .fillna(False)
    &
    benchmark_v3_working_df["source_url"]
    .astype("string")
    .str.contains(
        "voltnederland.org",
        case=False,
        na=False
    )
    &
    blank_or_missing_mask(
        benchmark_v3_working_df["institution"]
    )
)

volt_rows_to_update = int(
    volt_missing_metadata_mask.sum()
)

if volt_rows_to_update != 4:
    raise ValueError(
        "Expected four Volt records with missing institution metadata, "
        f"but found {volt_rows_to_update}."
    )

benchmark_v3_working_df.loc[
    volt_missing_metadata_mask,
    "party_name"
] = "Volt Nederland"

benchmark_v3_working_df.loc[
    volt_missing_metadata_mask,
    "institution"
] = "Volt Nederland"

benchmark_v3_working_df.loc[
    volt_missing_metadata_mask,
    "cleaning_notes"
] = (
    benchmark_v3_working_df.loc[
        volt_missing_metadata_mask,
        "cleaning_notes"
    ]
    .astype("string")
    .fillna("")
    .str.rstrip()
    + " Party name and institution completed from the official "
      "Volt Nederland source domain."
)

print(
    "Volt records updated:",
    volt_rows_to_update
)

display(
    benchmark_v3_working_df.loc[
        volt_missing_metadata_mask,
        [
            "benchmark_record_id",
            "person_name",
            "role_title",
            "party_name",
            "institution",
            "source_url"
        ]
    ]
)

Volt records updated: 4


,benchmark_record_id,person_name,role_title,party_name,institution,source_url
528,529,Denise Filippo,Interim covoorzitter,Volt Nederland,Volt Nederland,https://voltnederland.org/mensen/bestuur
561,562,Manou Zonderland,Bestuurslid,Volt Nederland,Volt Nederland,https://voltnederland.org/mensen/bestuur
567,568,Mazdak Soltani,Penningmeester,Volt Nederland,Volt Nederland,https://voltnederland.org/mensen/bestuur
590,591,Wesley Breel,Secretaris,Volt Nederland,Volt Nederland,https://voltnederland.org/mensen/bestuur


In [9]:
# -------------------------------------------------------------------
# Final benchmark-snapshot safety check
# -------------------------------------------------------------------
# The diplomatic source is dynamic. Therefore, expected category counts
# are derived from the actual raw component table in this documented run,
# minus the two auditable cleaning adjustments.

raw_category_counts_df = (
    raw_benchmark_components_df[
        "taxonomy_category"
    ]
    .value_counts(dropna=False)
    .rename_axis("taxonomy_category")
    .reset_index(name="raw_count")
)

removed_category_counts_df = (
    cleaning_adjustment_audit_df[
        "taxonomy_category"
    ]
    .value_counts(dropna=False)
    .rename_axis("taxonomy_category")
    .reset_index(name="documented_removed_count")
)

actual_final_counts_df = (
    benchmark_v3_working_df[
        "taxonomy_category"
    ]
    .value_counts(dropna=False)
    .rename_axis("taxonomy_category")
    .reset_index(name="actual_final_count")
)

final_count_check_df = (
    raw_category_counts_df
    .merge(
        removed_category_counts_df,
        on="taxonomy_category",
        how="outer"
    )
    .merge(
        actual_final_counts_df,
        on="taxonomy_category",
        how="outer"
    )
)

for column_name in [
    "raw_count",
    "documented_removed_count",
    "actual_final_count"
]:
    final_count_check_df[column_name] = (
        final_count_check_df[column_name]
        .fillna(0)
        .astype(int)
    )

final_count_check_df["expected_final_count"] = (
    final_count_check_df["raw_count"]
    - final_count_check_df["documented_removed_count"]
)

final_count_check_df["difference"] = (
    final_count_check_df["actual_final_count"]
    - final_count_check_df["expected_final_count"]
)

final_count_check_df = (
    final_count_check_df
    .sort_values("taxonomy_category")
    .reset_index(drop=True)
)

print("Category-count validation for the current live-source snapshot:")
display(final_count_check_df)

# The Senate list is a current complete institutional roster and must
# remain complete even when live diplomatic counts change.
senate_final_count = int(
    final_count_check_df.loc[
        final_count_check_df["taxonomy_category"].eq("senate_members"),
        "actual_final_count"
    ].sum()
)

if senate_final_count != 75:
    raise ValueError(
        "The final benchmark must retain 75 Senate members, but "
        f"contains {senate_final_count}."
    )

expected_final_snapshot_record_count = int(
    final_count_check_df["expected_final_count"].sum()
)

actual_final_snapshot_record_count = int(
    final_count_check_df["actual_final_count"].sum()
)

if actual_final_snapshot_record_count != expected_final_snapshot_record_count:
    raise ValueError(
        "Final benchmark category counts do not match the current raw "
        "collection snapshot minus documented adjustments. "
        f"Expected {expected_final_snapshot_record_count}, but found "
        f"{actual_final_snapshot_record_count}."
    )

if final_count_check_df["difference"].ne(0).any():
    raise ValueError(
        "At least one final category count differs from the current "
        "raw snapshot after documented cleaning adjustments."
    )

print(
    "\n✓ Final live-source benchmark validated: "
    f"{actual_final_snapshot_record_count} records."
)


Category-count validation for the current live-source snapshot:


,taxonomy_category,raw_count,documented_removed_count,actual_final_count,expected_final_count,difference
0,ambassadors,82,0,82,82,0
1,central_bank_board,5,0,5,5,0
2,charges_d_affaires,7,0,7,7,0
3,court_of_audit,3,0,3,3,0
4,head_of_state,1,0,1,1,0
5,high_judiciary_cbb,46,0,46,46,0
6,high_judiciary_crvb,88,2,86,86,0
7,high_judiciary_hr,34,0,34,34,0
8,high_judiciary_rvs,74,0,74,74,0
9,house_members,150,0,150,150,0



✓ Final live-source benchmark validated: 675 records.


## 4. Final benchmark quality checks and export

This section performs final quality checks on the combined benchmark and saves the benchmark used in the OpenSanctions matching analysis.

The final benchmark excludes one explicitly non-judicial CRvB board-member record because the operationalisation of the CRvB category is restricted to members with judicial duties. It also retains one record per person and taxonomy category, removing one duplicate CRvB person-category record.

The final benchmark size is snapshot-specific. It equals the number of records produced by the current documented collection run, minus these two documented cleaning adjustments.


In [10]:
# -------------------------------------------------------------------
# Final benchmark quality checks
# -------------------------------------------------------------------
# These checks identify missing values, duplicate person-category
# records, and suspicious names before the benchmark is exported.

required_benchmark_fields = [
    "benchmark_record_id",
    "person_name",
    "normalised_name",
    "taxonomy_category",
    "role_title",
    "institution",
    "source_url",
    "source_component"
]

quality_check_rows = []

for column_name in required_benchmark_fields:
    missing_count = int(
        blank_or_missing_mask(
            benchmark_v3_working_df[column_name]
        ).sum()
    )

    quality_check_rows.append({
        "check_type": "missing_required_value",
        "field": column_name,
        "result": missing_count,
        "status": "pass" if missing_count == 0 else "review"
    })

duplicate_person_category_count = int(
    benchmark_v3_working_df.duplicated(
        subset=[
            "normalised_name",
            "taxonomy_category"
        ],
        keep=False
    ).sum()
)

quality_check_rows.append({
    "check_type": "duplicate_person_category",
    "field": "normalised_name + taxonomy_category",
    "result": duplicate_person_category_count,
    "status": (
        "pass"
        if duplicate_person_category_count == 0
        else "review"
    )
})

empty_name_count = int(
    blank_or_missing_mask(
        benchmark_v3_working_df["person_name"]
    ).sum()
)

quality_check_rows.append({
    "check_type": "empty_person_name",
    "field": "person_name",
    "result": empty_name_count,
    "status": "pass" if empty_name_count == 0 else "review"
})

benchmark_quality_checks_df = pd.DataFrame(
    quality_check_rows
)

display(benchmark_quality_checks_df)

print("\nBenchmark records:", len(benchmark_v3_working_df))

print("\nRecords by taxonomy category:")
benchmark_category_counts_df = (
    benchmark_v3_working_df[
        "taxonomy_category"
    ]
    .value_counts()
    .rename_axis("taxonomy_category")
    .reset_index(name="record_count")
)

display(benchmark_category_counts_df)

print("\nRecords by source component:")
display(
    benchmark_v3_working_df[
        "source_component"
    ]
    .value_counts()
    .rename_axis("source_component")
    .reset_index(name="record_count")
)

,check_type,field,result,status
0,missing_required_value,benchmark_record_id,0,pass
1,missing_required_value,person_name,0,pass
2,missing_required_value,normalised_name,0,pass
3,missing_required_value,taxonomy_category,0,pass
4,missing_required_value,role_title,0,pass
5,missing_required_value,institution,0,pass
6,missing_required_value,source_url,0,pass
7,missing_required_value,source_component,0,pass
8,duplicate_person_category,normalised_name + taxonomy_category,0,pass
9,empty_person_name,person_name,0,pass



Benchmark records: 675

Records by taxonomy category:


,taxonomy_category,record_count
0,house_members,150
1,high_judiciary_crvb,86
2,ambassadors,82
3,registered_party_board,79
4,senate_members,75
5,high_judiciary_rvs,74
6,high_judiciary_cbb,46
7,high_judiciary_hr,34
8,national_executive,28
9,charges_d_affaires,7



Records by source component:


,source_component,record_count
0,curated_v1_baseline,286
1,diplomatic_selenium_extraction,89
2,manual_crvb_additions,86
3,manual_category_c_party_board_coding,79
4,senate_source_specific_extraction,75
5,manual_cbb_additions,46
6,manual_small_official_additions,9
7,manual_senior_military_additions,5


In [11]:
# -------------------------------------------------------------------
# Save final benchmark and quality-check outputs
# -------------------------------------------------------------------

final_benchmark_output_path = write_csv_safely(
    benchmark_v3_working_df,
    FINAL_BENCHMARK_PATH
)

quality_checks_output_path = write_csv_safely(
    benchmark_quality_checks_df,
    BENCHMARK_QUALITY_CHECKS_PATH
)

category_counts_output_path = write_csv_safely(
    benchmark_category_counts_df,
    BENCHMARK_CATEGORY_COUNTS_PATH
)

print("Saved final benchmark:")
print(final_benchmark_output_path)

print("\nSaved quality checks:")
print(quality_checks_output_path)

print("\nSaved category counts:")
print(category_counts_output_path)


print(
    "\nBenchmark export complete: "
    f"{len(benchmark_v3_working_df)} records were saved for the "
    "current live-source snapshot."
)


Saved final benchmark:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\main_current_benchmark_v3_extended.csv

Saved quality checks:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\benchmark_v3_quality_checks.csv

Saved category counts:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\benchmark_v3_category_counts.csv

Benchmark export complete: 675 records were saved for the current live-source snapshot.


In [12]:
# -------------------------------------------------------------------
# Locate the intended non-judicial CRvB exclusion row
# -------------------------------------------------------------------

crvb_raw_rows_df = raw_benchmark_components_df[
    (
        raw_benchmark_components_df["taxonomy_category"]
        .astype("string")
        .eq("high_judiciary_crvb")
        .fillna(False)
    )
].copy()

storm_or_board_candidates_df = crvb_raw_rows_df[
    (
        crvb_raw_rows_df["person_name"]
        .astype("string")
        .str.contains("Storm", case=False, na=False)
    )
    |
    (
        crvb_raw_rows_df["role_title"]
        .astype("string")
        .str.contains(
            "niet.rechterlijk|bestuur",
            case=False,
            na=False,
            regex=True
        )
    )
].copy()

print("Possible non-judicial CRvB rows found:", len(storm_or_board_candidates_df))

display(
    storm_or_board_candidates_df[
        [
            "person_name",
            "role_title",
            "taxonomy_category",
            "source_component",
            "cleaning_notes"
        ]
    ]
)

Possible non-judicial CRvB rows found: 2


,person_name,role_title,taxonomy_category,source_component,cleaning_notes
506,mr. R.W.L. Koopmans,Interim rechterlijk bestuurslid,high_judiciary_crvb,manual_crvb_additions,Reviewed official-source addition from the CRv...
507,drs. E. Storm,Niet-rechterlijk bestuurslid,high_judiciary_crvb,manual_crvb_additions,Reviewed official-source addition from the CRv...


In [13]:
# -------------------------------------------------------------------
# Inspect retained CRvB board-role records for later review
# -------------------------------------------------------------------

crvb_board_roles_df = benchmark_v3_working_df[
    (
        benchmark_v3_working_df["taxonomy_category"]
        .astype("string")
        .eq("high_judiciary_crvb")
        .fillna(False)
    )
    &
    (
        benchmark_v3_working_df["role_title"]
        .astype("string")
        .str.contains(
            "bestuurslid",
            case=False,
            na=False
        )
    )
].copy()

display(
    crvb_board_roles_df[
        [
            "benchmark_record_id",
            "person_name",
            "role_title",
            "source_component",
            "cleaning_notes"
        ]
    ]
)

,benchmark_record_id,person_name,role_title,source_component,cleaning_notes
216,217,R.W.L. Koopmans,Interim rechterlijk bestuurslid,manual_crvb_additions,Reviewed official-source addition from the CRv...


In [14]:
# -------------------------------------------------------------------
# Inspect rows with a missing institution value
# -------------------------------------------------------------------
# We inspect these records before filling anything automatically.

missing_institution_rows_df = benchmark_v3_working_df[
    blank_or_missing_mask(
        benchmark_v3_working_df["institution"]
    )
].copy()

print("Rows with a missing institution:", len(missing_institution_rows_df))

display(
    missing_institution_rows_df[
        [
            "benchmark_record_id",
            "person_name",
            "role_title",
            "taxonomy_category",
            "party_name",
            "source_component",
            "source_url",
            "cleaning_notes"
        ]
    ]
)

Rows with a missing institution: 0


,benchmark_record_id,person_name,role_title,taxonomy_category,party_name,source_component,source_url,cleaning_notes


In [15]:
# -------------------------------------------------------------------
# Validate that surname endings have not been stripped as degrees
# -------------------------------------------------------------------

name_cleaning_check_df = benchmark_v3_working_df[
    benchmark_v3_working_df["person_name_original"]
    .astype("string")
    .str.contains(
        r"Sjoerdsma|Heerma|Bosma|Boomsma|Paulusma|Wiersma",
        case=False,
        na=False,
        regex=True
    )
][
    [
        "person_name_original",
        "person_name",
        "taxonomy_category",
        "source_component"
    ]
].copy()

display(name_cleaning_check_df)

,person_name_original,person_name,taxonomy_category,source_component
288,Mr. G.J.J. (Guus) Heerma van Voss,G.J.J. (Guus) Heerma van Voss,high_judiciary_rvs,curated_v1_baseline
361,Diederik Boomsma,Diederik Boomsma,house_members,curated_v1_baseline
381,Femke Wiersma,Femke Wiersma,house_members,curated_v1_baseline
436,Martin Bosma,Martin Bosma,house_members,curated_v1_baseline
486,Wieke Paulusma,Wieke Paulusma,house_members,curated_v1_baseline
504,Pieter Heerma,Pieter Heerma,national_executive,curated_v1_baseline
509,Sjoerd Sjoerdsma,Sjoerd Sjoerdsma,national_executive,curated_v1_baseline
